# 📺 YouTube Playlist → HD Screenshots PDF
Downloads each video in **1080p HD**, captures sharp frames every N seconds, and builds a high-quality PDF.

**Output:**
- Full HD (1920×1080) screenshots
- One combined PDF with cover, TOC, and all video frames
- Timestamp label under every screenshot

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Install dependencies
# ─────────────────────────────────────────────────────────
!pip install -q yt-dlp reportlab Pillow
!ffmpeg -version | head -1
print("✅ All dependencies ready")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — ✏️ Configuration
# ─────────────────────────────────────────────────────────

PLAYLIST_URL         = "https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID"  # ← Paste here
OUTPUT_PDF           = "/content/playlist_HD_screenshots.pdf"

# Screenshot interval: one frame every N seconds
FRAME_EVERY_SECS     = 60        # 60 = 1 shot/min | 30 = 1 shot/30s

# Max screenshots per video
MAX_FRAMES_PER_VIDEO = 15

# HD video quality — downloads full 1080p
# 'bestvideo[height<=1080]+bestaudio/best[height<=1080]' = 1080p
# 'bestvideo[height<=720]+bestaudio/best[height<=720]'   = 720p
QUALITY              = 'bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/bestvideo[height<=1080]+bestaudio/best[height<=1080]/best'

# Screenshot JPEG quality (95 = near lossless, 75 = smaller file)
JPEG_QUALITY         = 95

# Output screenshot resolution (width px, height auto from 16:9)
FRAME_WIDTH          = 1920

# Limit videos for testing (None = all)
MAX_VIDEOS           = None

# Google Drive backup (set True to also save PDF to your Drive)
SAVE_TO_DRIVE        = False

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3 — Mount Google Drive (optional but recommended for long playlists)
# ─────────────────────────────────────────────────────────
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_PDF = "/content/drive/MyDrive/playlist_HD_screenshots.pdf"
    print(f"✅ Drive mounted. PDF will be saved to: {OUTPUT_PDF}")
else:
    print("ℹ️  Drive not mounted. PDF will be in /content/ (download in Cell 7)")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Fetch playlist metadata
# ─────────────────────────────────────────────────────────
import yt_dlp, os, subprocess, shutil, datetime

WORK_DIR = "/content/yt_hd_frames"
os.makedirs(WORK_DIR, exist_ok=True)

def fetch_playlist_info(url, max_videos=None):
    ydl_opts = {"quiet": True, "extract_flat": True, "playlist_end": max_videos}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
    videos = []
    for e in info.get("entries", []):
        if e and e.get("id"):
            videos.append({
                "title"    : e.get("title", "Untitled"),
                "video_id" : e["id"],
                "url"      : f"https://www.youtube.com/watch?v={e['id']}",
                "duration" : e.get("duration") or 0,
                "uploader" : e.get("uploader", ""),
            })
    playlist_title = info.get("title", "YouTube Playlist")
    print(f"✅ Found {len(videos)} videos in '{playlist_title}'")
    return playlist_title, videos


playlist_title, videos = fetch_playlist_info(PLAYLIST_URL, MAX_VIDEOS)
print(f"\nFirst 5 titles:")
for v in videos[:5]:
    print(f"  • {v['title']}")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Download HD videos & extract frames
# ─────────────────────────────────────────────────────────

def format_time(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s   = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"


def get_duration(vid_path):
    """Use ffprobe to get exact video duration in seconds."""
    try:
        r = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration",
             "-of", "default=noprint_wrappers=1:nokey=1", vid_path],
            capture_output=True, text=True, timeout=30
        )
        return float(r.stdout.strip())
    except Exception:
        return 0


def extract_frame(vid_path, timestamp, out_path, width=1920, jpeg_q=95):
    """
    Extract one HD frame at `timestamp` seconds.
    Uses -q:v 1 (best quality) and scales to `width` px wide.
    """
    cmd = [
        "ffmpeg",
        "-ss", str(timestamp),       # seek before input (fast)
        "-i", vid_path,
        "-vframes", "1",             # grab exactly 1 frame
        "-vf", f"scale={width}:-1",  # HD width, auto height
        "-q:v", "1",                 # highest JPEG quality (1=best, 31=worst)
        "-pix_fmt", "yuvj420p",      # full color range
        out_path,
        "-y", "-loglevel", "error"
    ]
    subprocess.run(cmd, capture_output=True, timeout=120)
    return os.path.exists(out_path)


def process_video(video, idx, frame_every, max_frames, quality, frame_width, jpeg_q):
    vid_dir  = os.path.join(WORK_DIR, f"{idx:03d}")
    os.makedirs(vid_dir, exist_ok=True)
    vid_path = os.path.join(vid_dir, "video.mp4")

    # ── Download HD ──────────────────────────────────────
    ydl_opts = {
        "quiet"       : True,
        "format"      : quality,
        "outtmpl"     : vid_path,
        "noplaylist"  : True,
        "merge_output_format": "mp4",
        "postprocessors": [{
            "key": "FFmpegVideoConvertor",
            "preferedformat": "mp4",
        }],
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([video['url']])
    except Exception as e:
        print(f"    ✗ Download error: {str(e)[:80]}")
        return []

    # Find the downloaded file (extension might vary)
    if not os.path.exists(vid_path):
        candidates = sorted([
            os.path.join(vid_dir, f)
            for f in os.listdir(vid_dir)
            if not f.endswith('.jpg')
        ], key=os.path.getsize, reverse=True)
        if not candidates:
            print("    ✗ No video file found after download")
            return []
        vid_path = candidates[0]

    file_mb = os.path.getsize(vid_path) / 1024 / 1024
    print(f"    📦 Downloaded: {file_mb:.0f} MB")

    # ── Get real duration ─────────────────────────────────
    duration = get_duration(vid_path) or video.get('duration', 300)

    # ── Compute timestamps ────────────────────────────────
    # Always grab frame at 5s (opening slide), then every N seconds
    timestamps = [5.0]
    t = frame_every
    while t < duration - 5 and len(timestamps) < max_frames:
        timestamps.append(float(t))
        t += frame_every
    timestamps = timestamps[:max_frames]

    # ── Extract frames ────────────────────────────────────
    frames = []
    for ts in timestamps:
        out_img = os.path.join(vid_dir, f"frame_{int(ts):06d}.jpg")
        ok = extract_frame(vid_path, ts, out_img, frame_width, jpeg_q)
        if ok:
            frames.append((out_img, format_time(ts)))

    # ── Delete video to free disk space ──────────────────
    try:
        os.remove(vid_path)
    except Exception:
        pass

    return frames


# ── Main processing loop ─────────────────────────────────
print(f"🎬 Processing {len(videos)} videos in HD (1080p)")
print(f"   Frame every {FRAME_EVERY_SECS}s | Max {MAX_FRAMES_PER_VIDEO} per video | JPEG Q={JPEG_QUALITY}\n")

total_frames = 0
for i, v in enumerate(videos, 1):
    print(f"[{i:02d}/{len(videos)}] {v['title'][:72]}")
    frames = process_video(
        v, i,
        FRAME_EVERY_SECS,
        MAX_FRAMES_PER_VIDEO,
        QUALITY,
        FRAME_WIDTH,
        JPEG_QUALITY
    )
    v['frames'] = frames
    total_frames += len(frames)
    print(f"    ✓ {len(frames)} HD frames captured | Total so far: {total_frames}")

print(f"\n✅ Done! {total_frames} total HD frames from {len(videos)} videos.")

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6 — Build the HD PDF
# ─────────────────────────────────────────────────────────
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    PageBreak, HRFlowable, Image
)
from reportlab.lib.enums import TA_CENTER
from PIL import Image as PILImage

# Use landscape A4 so wide 16:9 screenshots fill the page
PAGE_W, PAGE_H = landscape(A4)   # 842 x 595 pts
MARGIN    = 1.2 * cm
CONTENT_W = PAGE_W - 2 * MARGIN
CONTENT_H = PAGE_H - 3.5 * cm   # leave room for title + timestamp

styles = getSampleStyleSheet()
RED    = colors.HexColor("#FF0000")
DARK   = colors.HexColor("#111111")
GREY   = colors.HexColor("#666666")

s_cover = ParagraphStyle("Cover", parent=styles["Title"],
            fontSize=28, textColor=DARK, alignment=TA_CENTER, leading=34)
s_sub   = ParagraphStyle("Sub",   parent=styles["Normal"],
            fontSize=13, textColor=GREY, alignment=TA_CENTER, spaceAfter=5)
s_toc_h = ParagraphStyle("TocH", parent=styles["Heading1"],
            fontSize=17, textColor=RED, spaceAfter=6)
s_toc_e = ParagraphStyle("TocE", parent=styles["Normal"],
            fontSize=10, leading=18, leftIndent=10)
s_vid_h = ParagraphStyle("VH",   parent=styles["Heading2"],
            fontSize=13, textColor=RED, spaceBefore=3, spaceAfter=2)
s_meta  = ParagraphStyle("Meta", parent=styles["Normal"],
            fontSize=8.5, textColor=GREY, spaceAfter=4)
s_ts    = ParagraphStyle("TS",   parent=styles["Normal"],
            fontSize=8, textColor=GREY, alignment=TA_CENTER, spaceBefore=1)
s_notx  = ParagraphStyle("NoTx", parent=styles["Italic"],
            fontSize=11, textColor=GREY, alignment=TA_CENTER, spaceBefore=30)

def esc(t):
    return str(t).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
def p(text, style):
    return Paragraph(esc(text), style)
def fmt_dur(secs):
    if not secs: return "N/A"
    h, r = divmod(int(secs), 3600); m, s = divmod(r, 60)
    return f"{h}h {m}m {s}s" if h else f"{m}m {s}s"

def fit_image(img_path, max_w, max_h):
    """Scale image to fit within max_w x max_h preserving aspect ratio."""
    try:
        with PILImage.open(img_path) as im:
            iw, ih = im.size
        ratio  = min(max_w / iw, max_h / ih)
        return Image(img_path, width=iw * ratio, height=ih * ratio)
    except Exception as e:
        return p(f"[Image load error: {e}]", s_notx)


def build_pdf(title, videos, path):
    doc = SimpleDocTemplate(
        path, pagesize=landscape(A4),
        leftMargin=MARGIN, rightMargin=MARGIN,
        topMargin=1.5*cm, bottomMargin=1.5*cm,
        title=title
    )
    story = []

    # ── Cover ─────────────────────────────────────────────
    total_frames = sum(len(v.get('frames',[])) for v in videos)
    story += [
        Spacer(1, 3*cm),
        p("📺  " + title, s_cover),
        Spacer(1, .6*cm),
        p(f"{len(videos)} Videos  •  {total_frames} HD Screenshots", s_sub),
        p(f"Generated on {datetime.date.today().strftime('%B %d, %Y')}", s_sub),
        PageBreak()
    ]

    # ── Table of Contents ─────────────────────────────────
    story += [
        p("Table of Contents", s_toc_h),
        HRFlowable(width="100%", thickness=1, color=RED, spaceAfter=8)
    ]
    for i, v in enumerate(videos, 1):
        n = len(v.get('frames', []))
        story.append(p(f"{i}.  {v['title']}  [{n} screenshot{'s' if n!=1 else ''}]", s_toc_e))
    story.append(PageBreak())

    # ── Per-video sections ────────────────────────────────
    for i, v in enumerate(videos, 1):
        story += [
            p(f"{i}. {v['title']}", s_vid_h),
            HRFlowable(width="100%", thickness=0.6, color=RED, spaceAfter=3),
            p(f"🔗 {v['url']}   ⏱ {fmt_dur(v.get('duration'))}", s_meta)
        ]

        frames = v.get('frames', [])
        if frames:
            for img_path, ts_label in frames:
                img_el = fit_image(img_path, CONTENT_W, CONTENT_H)
                story.append(img_el)
                story.append(p(f"⏱  {ts_label}", s_ts))
                story.append(PageBreak())    # each screenshot on its own landscape page
        else:
            story += [p("No frames extracted for this video.", s_notx), PageBreak()]

    doc.build(story)
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f"✅ PDF saved → {path}")
    print(f"   Size: {size_mb:.1f} MB  |  Pages: ~{total_frames + len(videos) + 2}")


build_pdf(playlist_title, videos, OUTPUT_PDF)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7 — Clean up & Download PDF
# ─────────────────────────────────────────────────────────
import shutil
shutil.rmtree(WORK_DIR, ignore_errors=True)
print("🗑️  Temp files deleted")

if not SAVE_TO_DRIVE:
    from google.colab import files
    files.download(OUTPUT_PDF)
else:
    print(f"✅ PDF already saved to your Google Drive: {OUTPUT_PDF}")

---
## 📋 Settings Reference

| Setting | Value | Meaning |
|---|---|---|
| `FRAME_EVERY_SECS` | `60` | 1 screenshot per minute |
| `FRAME_EVERY_SECS` | `30` | 1 screenshot every 30 seconds |
| `MAX_FRAMES_PER_VIDEO` | `15` | Max 15 shots per video |
| `QUALITY` | `1080p` | Full HD download |
| `QUALITY` | `720p` | Slightly faster, still HD |
| `JPEG_QUALITY` | `95` | Near-lossless screenshots |
| `FRAME_WIDTH` | `1920` | Full HD width |
| `SAVE_TO_DRIVE` | `True` | Auto-save to Google Drive (recommended for 95 videos!) |

## ⚠️ Tips for 95 Videos
- Set `SAVE_TO_DRIVE = True` — Colab sessions can disconnect and you'll lose the PDF
- Colab free tier has a ~12h session limit — use **Colab Pro** or run overnight
- Expected PDF size: **500 MB – 2 GB** depending on settings
- If you get a disk space error, reduce `MAX_FRAMES_PER_VIDEO` to 8